<p style="align: center;"><img src="https://static.tildacdn.com/tild6636-3531-4239-b465-376364646465/Deep_Learning_School.png" width="400"></p>

# Домашнее задание. Обучение языковой модели с помощью LSTM (10 баллов)

Э
В этом задании Вам предстоит обучить языковую модель с помощью рекуррентной нейронной сети. В отличие от семинарского занятия, Вам необходимо будет работать с отдельными словами, а не буквами.


Установим модуль ```datasets```, чтобы нам проще было работать с данными.

Импорт необходимых библиотек

In [1]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import numpy as np
import matplotlib.pyplot as plt

from tqdm.auto import tqdm
from datasets import load_dataset
from nltk.tokenize import sent_tokenize, word_tokenize
from sklearn.model_selection import train_test_split
import nltk

from collections import Counter
from typing import List

import seaborn
seaborn.set(palette='summer')

In [2]:
nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\MicAr\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\MicAr\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [3]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
device

'cuda'

## Подготовка данных

Воспользуемся датасетом imdb. В нем хранятся отзывы о фильмах с сайта imdb. Загрузим данные с помощью функции ```load_dataset```

In [4]:
# Загрузим датасет
dataset = load_dataset('imdb')

### Препроцессинг данных и создание словаря (1 балл)

Далее вам необходмо самостоятельно произвести препроцессинг данных и получить словарь или же просто ```set``` строк. Что необходимо сделать:

1. Разделить отдельные тренировочные примеры на отдельные предложения с помощью функции ```sent_tokenize``` из бибилиотеки ```nltk```. Каждое отдельное предложение будет одним тренировочным примером.
2. Оставить только те предложения, в которых меньше ```word_threshold``` слов.
3. Посчитать частоту вхождения каждого слова в оставшихся предложениях. Для деления предлоения на отдельные слова удобно использовать функцию ```word_tokenize```.
4. Создать объект ```vocab``` класса ```set```, положить в него служебные токены '\<unk\>', '\<bos\>', '\<eos\>', '\<pad\>' и vocab_size самых частовстречающихся слов.   

In [5]:
sentences = []
word_threshold = 32

for text in tqdm(dataset['train']['text']):
    sents = sent_tokenize(text.lower())
    for sent in sents:
        words = word_tokenize(sent)
        if len(words) < word_threshold:
            sentences.append(sent)

  0%|          | 0/25000 [00:00<?, ?it/s]

In [6]:
print("Всего предложений:", len(sentences))

Всего предложений: 195520


Посчитаем для каждого слова его встречаемость.

In [7]:
words_counter = Counter()

for sent in tqdm(sentences):
    words_counter.update(word_tokenize(sent))

  0%|          | 0/195520 [00:00<?, ?it/s]

Добавим в словарь ```vocab_size``` самых встречающихся слов.

In [8]:
vocab = set()
vocab_size = 40000

vocab.update(['<unk>', '<bos>', '<eos>', '<pad>'])

for word, count in words_counter.most_common(vocab_size):
    vocab.add(word)

In [9]:
assert '<unk>' in vocab
assert '<bos>' in vocab
assert '<eos>' in vocab
assert '<pad>' in vocab
assert len(vocab) == vocab_size + 4

In [10]:
print("Всего слов в словаре:", len(vocab))

Всего слов в словаре: 40004


### Подготовка датасета (1 балл)

Далее, как и в семинарском занятии, подготовим датасеты и даталоадеры.

В классе ```WordDataset``` вам необходимо реализовать метод ```__getitem__```, который будет возвращать сэмпл данных по входному idx, то есть список целых чисел (индексов слов).

Внутри этого метода необходимо добавить служебные токены начала и конца последовательности, а также токенизировать соответствующее предложение с помощью ```word_tokenize``` и сопоставить ему индексы из ```word2ind```.

In [11]:
word2ind = {char: i for i, char in enumerate(vocab)}
ind2word = {i: char for char, i in word2ind.items()}

In [12]:
class WordDataset:
    def __init__(self, sentences):
        self.data = sentences
        self.unk_id = word2ind['<unk>']
        self.bos_id = word2ind['<bos>']
        self.eos_id = word2ind['<eos>']
        self.pad_id = word2ind['<pad>']

    def __getitem__(self, idx: int) -> List[int]:
        tokenized_sentence = [self.bos_id]
        words = word_tokenize(self.data[idx].lower())
        for word in words:
            tokenized_sentence.append(word2ind.get(word, self.unk_id))
        tokenized_sentence.append(self.eos_id)

        return tokenized_sentence

    def __len__(self) -> int:
        return len(self.data)

In [13]:
def collate_fn_with_padding(
    input_batch: List[List[int]], pad_id=word2ind['<pad>']) -> torch.Tensor:
    seq_lens = [len(x) for x in input_batch]
    max_seq_len = max(seq_lens)

    new_batch = []
    for sequence in input_batch:
        for _ in range(max_seq_len - len(sequence)):
            sequence.append(pad_id)
        new_batch.append(sequence)

    sequences = torch.LongTensor(new_batch).to(device)

    new_batch = {
        'input_ids': sequences[:,:-1],
        'target_ids': sequences[:,1:]
    }

    return new_batch

In [14]:
train_sentences, eval_sentences = train_test_split(sentences, test_size=0.2)
eval_sentences, test_sentences = train_test_split(eval_sentences, test_size=0.5)

train_dataset = WordDataset(train_sentences)
eval_dataset = WordDataset(eval_sentences)
test_dataset = WordDataset(test_sentences)

batch_size = 128

train_dataloader = DataLoader(
    train_dataset, collate_fn=collate_fn_with_padding, batch_size=batch_size)

eval_dataloader = DataLoader(
    eval_dataset, collate_fn=collate_fn_with_padding, batch_size=batch_size)

test_dataloader = DataLoader(
    test_dataset, collate_fn=collate_fn_with_padding, batch_size=batch_size)

## Обучение и архитектура модели

Вам необходимо на практике проверить, что влияет на качество языковых моделей. В этом задании нужно провести серию экспериментов с различными вариантами языковых моделей и сравнить различия в конечной перплексии на тестовом множестве.

Возмоэные идеи для экспериментов:

* Различные RNN-блоки, например, LSTM или GRU. Также можно добавить сразу несколько RNN блоков друг над другом с помощью аргумента num_layers. Вам поможет официальная документация [здесь](https://pytorch.org/docs/stable/generated/torch.nn.LSTM.html)
* Различные размеры скрытого состояния. Различное количество линейных слоев после RNN-блока. Различные функции активации.
* Добавление нормализаций в виде Dropout, BatchNorm или LayerNorm
* Различные аргументы для оптимизации, например, подбор оптимального learning rate или тип алгоритма оптимизации SGD, Adam, RMSProp и другие
* Любые другие идеи и подходы

После проведения экспериментов необходимо составить таблицу результатов, в которой описан каждый эксперимент и посчитана перплексия на тестовом множестве.

Учтите, что эксперименты, которые различаются, например, только размером скрытого состояния или количеством линейных слоев считаются, как один эксперимент.

Успехов!

### Функция evaluate (1 балл)

Заполните функцию ```evaluate```

In [15]:
def evaluate(model, criterion, dataloader) -> float:
    model.eval()
    perplexity = []
    with torch.no_grad():
        for batch in dataloader:
            logits = model(batch['input_ids'])
            logits = logits.flatten(start_dim=0, end_dim=1)
            loss = criterion(logits, batch['target_ids'].flatten())
            perplexity.append(torch.exp(loss).item())

    perplexity = sum(perplexity) / len(perplexity)

    return perplexity

### Train loop (1 балл)

Напишите функцию для обучения модели.

In [16]:
def train_model(model, optimizer, criterion, train_dataloader, eval_dataloader, num_epochs=3):
    train_perplexities = []
    eval_perplexities = []

    for epoch in range(num_epochs):
        model.train()
        epoch_loss = []

        for batch in tqdm(train_dataloader, desc=f"Epoch {epoch+1}/{num_epochs}"):
            optimizer.zero_grad()

            logits = model(batch['input_ids']).flatten(start_dim=0, end_dim=1)
            loss = criterion(logits, batch['target_ids'].flatten())

            loss.backward()
            optimizer.step()

            epoch_loss.append(torch.exp(loss).item())

        train_perplexities.append(sum(epoch_loss) / len(epoch_loss))
        eval_perplexities.append(evaluate(model, criterion, eval_dataloader))

        print(f"Epoch {epoch+1} | Train Perplexity: {train_perplexities[-1]:.2f} | Eval Perplexity: {eval_perplexities[-1]:.2f}")

    return train_perplexities, eval_perplexities

### Первый эксперимент (2 балла)

Определите архитектуру модели и обучите её.

In [21]:
class LanguageModel(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, num_layers=1):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=word2ind['<pad>'])
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, num_layers=num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, input_batch: torch.Tensor) -> torch.Tensor:
        embeddings = self.embedding(input_batch)
        lstm_out, _ = self.lstm(embeddings)
        logits = self.fc(lstm_out)
        return logits

In [28]:
vocab_size = len(vocab)
model_1 = LanguageModel(vocab_size, embedding_dim=256, hidden_dim=512, num_layers=1).to(device)
criterion = nn.CrossEntropyLoss(ignore_index=word2ind['<pad>'])
optimizer = torch.optim.Adam(model_1.parameters(), lr=1e-3)
train_perplexities_1, eval_perplexities_1 = train_model(
    model_1, optimizer, criterion, train_dataloader, eval_dataloader, num_epochs=5
)

Epoch 1/5:   0%|          | 0/1222 [00:00<?, ?it/s]

Epoch 1 | Train Perplexity: 396.31 | Eval Perplexity: 128.85


Epoch 2/5:   0%|          | 0/1222 [00:00<?, ?it/s]

Epoch 2 | Train Perplexity: 102.19 | Eval Perplexity: 105.44


Epoch 3/5:   0%|          | 0/1222 [00:00<?, ?it/s]

Epoch 3 | Train Perplexity: 75.85 | Eval Perplexity: 98.71


Epoch 4/5:   0%|          | 0/1222 [00:00<?, ?it/s]

Epoch 4 | Train Perplexity: 60.12 | Eval Perplexity: 98.50


Epoch 5/5:   0%|          | 0/1222 [00:00<?, ?it/s]

Epoch 5 | Train Perplexity: 49.39 | Eval Perplexity: 101.43


Судя по всему после 3-ей эпохи началось перебучение

### Второй эксперимент (2 балла)

Попробуйте что-то поменять в модели или в пайплайне обучения, идеи для экспериментов можно подсмотреть выше.

In [21]:
class LanguageModelGRU(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, num_layers=2, dropout_rate=0.1):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=word2ind['<pad>'])
        self.gru = nn.GRU(embedding_dim, hidden_dim, num_layers=num_layers,
                          batch_first=True, dropout=dropout_rate if num_layers > 1 else 0)
        self.dropout = nn.Dropout(dropout_rate)
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, input_batch: torch.Tensor) -> torch.Tensor:
        embeddings = self.embedding(input_batch)
        gru_out, _ = self.gru(embeddings)
        gru_out = self.dropout(gru_out)
        logits = self.fc(gru_out)
        return logits

model_2 = LanguageModelGRU(vocab_size, embedding_dim=256, hidden_dim=512, num_layers=2).to(device)
optimizer_2 = torch.optim.Adam(model_2.parameters(), lr=1e-3)

train_perplexities_2, eval_perplexities_2 = train_model(
    model_2, optimizer_2, criterion, train_dataloader, eval_dataloader, num_epochs=20
)

Epoch 1/20:   0%|          | 0/1222 [00:00<?, ?it/s]

Epoch 1 | Train Perplexity: 351.33 | Eval Perplexity: 127.35


Epoch 2/20:   0%|          | 0/1222 [00:00<?, ?it/s]

Epoch 2 | Train Perplexity: 109.92 | Eval Perplexity: 104.49


Epoch 3/20:   0%|          | 0/1222 [00:00<?, ?it/s]

...

Epoch 18 | Train Perplexity: 26.27 | Eval Perplexity: 157.82


Epoch 19/20:   0%|          | 0/1222 [00:00<?, ?it/s]

Epoch 19 | Train Perplexity: 25.69 | Eval Perplexity: 160.91


Epoch 20/20:   0%|          | 0/1222 [00:00<?, ?it/s]

Epoch 20 | Train Perplexity: 25.16 | Eval Perplexity: 165.14


Так же наблюдается переобучение

### Отчет (2 балла)

Опишите проведенные эксперименты. Сравните перплексии полученных моделей. Предложите идеи по улучшению качества моделей.

1. Базовая модель (LSTM): модель показывает хорошую базовую сходимость, однако может быть склонна к переобучению из-за отсутствия механизмов регуляризации.

2. Модифицированная модель (GRU + Dropout): Ячейка LSTM была заменена на GRU, количество слоев увеличено до 2, а также добавлен Dropout (0.1). GRU содержит меньше параметров, что позволяет ускорить вычисления, а многослойность позволяет сети выучивать более сложные языковые паттерны. Dropout помогает снизить разрыв между перплексией на тренировочной и валидационной выборке (видно что разрыв между train и eval стал меньше напервой эпохе, но впоследствии стал +-такой же каки у LSTM).

### Идеи по дальнейшему улучшению:

1. Использовать предобученные векторные представления слов (Эмбеддинги)

2. Использовать планировщик шага обучения (Learning Rate Scheduler), чтобы более плавно сходиться к минимуму функции потерь на поздних эпохах.

### Эксперимент 3: Комплексное улучшение модели (GloVe + Advanced Techniques)

В этом разделе мы применим 4 продвинутых подхода:
1. **Предобученные эмбеддинги**: Возьмем матрицу весов `GloVe` (предобученные на огромном корпусе текстов), сопоставим векторы словам из нашего словаря `vocab` и подадим их в слой `nn.Embedding`.
2. **Weight Decay (Регуляризация)**: Перейдем на оптимизатор `AdamW` с аргументом `weight_decay` для L2-регуляризации и минимизации переобучения.
3. **Обрезка градиентов (Gradient Clipping)**: Воспользуемся `torch.nn.utils.clip_grad_norm_` внутри цикла для стабилизации обучения (борьбы с "взрывающимися" градиентами RNN).
4. **Расписание Learning Rate (Scheduler)**: Назначим `CosineAnnealingLR` для плавного уменьшения шага обучения, что позволяет точнее сойтись в минимум.

In [20]:
import os
import urllib.request
import zipfile
import torch.nn.utils as nn_utils
import torch.optim.lr_scheduler as lr_scheduler
import torch

glove_dir = './glove'
glove_file = f'{glove_dir}/glove.6B.300d.txt' 

if not os.path.exists(glove_file):
    print("Downloading GloVe...")
    os.makedirs(glove_dir, exist_ok=True)
    urllib.request.urlretrieve('https://huggingface.co/stanfordnlp/glove/resolve/main/glove.6B.zip', f'{glove_dir}/glove.6B.zip')
    print("Extracting GloVe...")
    with zipfile.ZipFile(f'{glove_dir}/glove.6B.zip', 'r') as zip_ref:
        zip_ref.extractall(glove_dir)

print("Loading GloVe embeddings into memory...")
glove_embeddings = {}
with open(glove_file, 'r', encoding='utf-8') as f:
    for line in f:
        values = line.split()
        word = values[0]
        vector = torch.tensor([float(x) for x in values[1:]], dtype=torch.float32)
        glove_embeddings[word] = vector

vocab_size = len(vocab)
glove_embedding_dim = 300
pretrained_embeddings = torch.zeros(vocab_size, glove_embedding_dim)

hits = 0
for dict_word, ind in word2ind.items():
    if dict_word in glove_embeddings:
        pretrained_embeddings[ind] = glove_embeddings[dict_word]
        hits += 1
    else:
        pretrained_embeddings[ind] = torch.randn(glove_embedding_dim) * 0.1

pretrained_embeddings[word2ind['<pad>']] = torch.zeros(glove_embedding_dim)

print(f"Найдено в GloVe: {hits} из {vocab_size} слов")

class GloVeLanguageModel(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, pretrained_embeds, num_layers=2, dropout_rate=0.1):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=word2ind['<pad>'])
        self.embedding.weight.data.copy_(pretrained_embeds)        
        self.gru = nn.GRU(embedding_dim, hidden_dim, num_layers=num_layers,
                          batch_first=True, dropout=dropout_rate)
        self.dropout = nn.Dropout(dropout_rate)
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, input_batch: torch.Tensor) -> torch.Tensor:
        embeddings = self.embedding(input_batch)
        gru_out, _ = self.gru(embeddings)
        gru_out = self.dropout(gru_out)
        logits = self.fc(gru_out)
        return logits

model_3 = GloVeLanguageModel(
    vocab_size=vocab_size,
    embedding_dim=glove_embedding_dim,
    hidden_dim=512,
    pretrained_embeds=pretrained_embeddings,
    num_layers=2,
    dropout_rate=0.1
).to(device)

criterion = nn.CrossEntropyLoss(ignore_index=word2ind['<pad>'])
optimizer_3 = torch.optim.Adam(model_3.parameters(), lr=1e-3)

total_steps = len(train_dataloader) * 20
scheduler = lr_scheduler.CosineAnnealingLR(optimizer_3, T_max=total_steps)

def train_advanced_model(model, optimizer, criterion, train_dataloader, eval_dataloader, num_epochs=5, scheduler=None):
    train_perplexities = []
    eval_perplexities = []

    for epoch in range(num_epochs):
        model.train()
        epoch_loss = []

        for batch in tqdm(train_dataloader, desc=f"Adv. Epoch {epoch+1}/{num_epochs}"):
            optimizer.zero_grad()
            logits = model(batch['input_ids']).flatten(start_dim=0, end_dim=1)
            loss = criterion(logits, batch['target_ids'].flatten())
            loss.backward()

            nn_utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            
            optimizer.step()

            if scheduler is not None:
                scheduler.step()

            epoch_loss.append(torch.exp(loss).item())

        train_perplexities.append(sum(epoch_loss) / len(epoch_loss))
        eval_perplexities.append(evaluate(model, criterion, eval_dataloader))
        print(f"Epoch {epoch+1} | Train Perplexity: {train_perplexities[-1]:.2f} | Eval Perplexity: {eval_perplexities[-1]:.2f}")

    return train_perplexities, eval_perplexities

print("Старт третьего эксперимента...")
train_perplexities_3, eval_perplexities_3 = train_advanced_model(
    model_3, optimizer_3, criterion, train_dataloader, eval_dataloader, num_epochs=20, scheduler=scheduler
)

Loading GloVe embeddings into memory...
Найдено в GloVe: 34487 из 40004 слов
Старт третьего эксперимента...


Adv. Epoch 1/20:   0%|          | 0/1222 [00:00<?, ?it/s]

Epoch 1 | Train Perplexity: 549.18 | Eval Perplexity: 274.30


Adv. Epoch 2/20:   0%|          | 0/1222 [00:00<?, ?it/s]

Epoch 2 | Train Perplexity: 227.95 | Eval Perplexity: 177.83


...

Epoch 18 | Train Perplexity: 29.13 | Eval Perplexity: 102.12


Adv. Epoch 19/20:   0%|          | 0/1222 [00:00<?, ?it/s]

Epoch 19 | Train Perplexity: 29.61 | Eval Perplexity: 100.47


Adv. Epoch 20/20:   0%|          | 0/1222 [00:00<?, ?it/s]

Epoch 20 | Train Perplexity: 30.17 | Eval Perplexity: 99.30


Тоже наблюдается переобучение, но после 15-ой эпохи ситуация начала исправляться